# Notebook 06 – Model Improvement and Experiments

## Objective

The goal of this notebook is to improve the performance of the fine-tuned BERT model through systematic experiments.

Unlike the previous notebooks, this notebook focuses on comparing different training strategies and analyzing their impact on model performance.

Experiments include:

- Increasing the number of training epochs
- Comparing evaluation metrics
- Analyzing improvements
- Selecting the best-performing model

## Experiment 1 – Increasing the Number of Epochs

### Hypothesis

Training the model for more epochs may improve its ability to learn meaningful patterns from the training data, resulting in higher recall and F1-score.

However, excessive training may also increase the risk of overfitting.

## Experiment 1 – Additional Fine-Tuning

### Goal

Evaluate whether training the model for additional epochs improves the overall classification performance.

### Hypothesis

Training for three additional epochs will improve the model's ability to recognize complex language patterns, particularly in the Depression and Suicidal classes.

### Metrics

The following metrics will be compared with the baseline model:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

### Success Criteria

The experiment will be considered successful if the recall and F1-score of the Suicidal class improve without a significant decrease in overall performance.

## Load Previous Checkpoint

The best checkpoint from Notebook 04 is loaded to continue fine-tuning without restarting the training process.

Both the model parameters and optimizer state are restored.

In [1]:
import torch
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(device)

mps


In [3]:
NUM_LABELS = 4

MODEL_PATH = "../models/bert_mental_health"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.to(device)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!


In [4]:
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
)

## Rebuild Data Pipeline

The processed train, validation, and test datasets are reloaded to continue
fine-tuning and model improvement experiments.

The same label mapping, tokenizer configuration, and dataset structure used
during the original BERT fine-tuning are preserved for consistency.

In [8]:
from pathlib import Path
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

PROCESSED_DATA_DIR = Path("../data/processed")

train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
validation_df = pd.read_csv(PROCESSED_DATA_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")

print("Training set:", train_df.shape)
print("Validation set:", validation_df.shape)
print("Test set:", test_df.shape)

Training set: (34254, 2)
Validation set: (7340, 2)
Test set: (7341, 2)


In [10]:
label2id = {
    "Anxiety": 0,
    "Depression": 1,
    "Normal": 2,
    "Suicidal": 3,
}

id2label = {v: k for k, v in label2id.items()}

for dataframe in [train_df, validation_df, test_df]:
    dataframe["label"] = dataframe["status"].map(label2id)

print("label2id:", label2id)
print("id2label:", id2label)

label2id: {'Anxiety': 0, 'Depression': 1, 'Normal': 2, 'Suicidal': 3}
id2label: {0: 'Anxiety', 1: 'Depression', 2: 'Normal', 3: 'Suicidal'}


In [11]:
MODEL_PATH = "../models/bert_mental_health"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

MAX_LENGTH = 128

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [13]:
BATCH_SIZE = 16

train_dataset = MentalHealthDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

validation_dataset = MentalHealthDataset(
    dataframe=validation_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

test_dataset = MentalHealthDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

Train batches: 2141
Validation batches: 459
Test batches: 459


In [12]:
class MentalHealthDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length: int = 128,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        text = self.dataframe.iloc[index]["text"]
        label = self.dataframe.iloc[index]["label"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [14]:
batch = next(iter(train_loader))

print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape:", batch["labels"].shape)

print("First labels:", batch["labels"][:5])

input_ids shape: torch.Size([16, 128])
attention_mask shape: torch.Size([16, 128])
labels shape: torch.Size([16])
First labels: tensor([2, 2, 1, 2, 2])


In [15]:
start_epoch = 0
TOTAL_EPOCHS = 1

In [16]:
model.train()

for epoch in range(start_epoch, TOTAL_EPOCHS):

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}")
    print(f"Average Loss: {avg_loss:.4f}")

Epoch 1
Average Loss: 0.4756


In [17]:
print("start_epoch:", start_epoch)
print("TOTAL_EPOCHS:", TOTAL_EPOCHS)
print("Epoch values:", list(range(start_epoch, TOTAL_EPOCHS)))

start_epoch: 0
TOTAL_EPOCHS: 1
Epoch values: [0]


In [18]:
ADDITIONAL_EPOCHS = 2

model.train()

for epoch in range(ADDITIONAL_EPOCHS):

    total_loss = 0.0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Additional Epoch {epoch + 1}")
    print(f"Average Loss: {avg_loss:.4f}")

Additional Epoch 1
Average Loss: 0.3246
Additional Epoch 2
Average Loss: 0.2262


### Continued Fine-Tuning

The model had already completed one training epoch in the current session.

Therefore, two additional epochs were performed to reach approximately three epochs of total fine-tuning.

The objective is to evaluate whether continued training improves class-specific performance, especially recall and F1-score for the Suicidal class.

In [19]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1,
        )

        all_predictions.extend(predictions.cpu().numpy())

        all_labels.extend(labels.cpu().numpy())

print("Evaluation completed.")
print("Predictions:", len(all_predictions))

Evaluation completed.
Predictions: 7341


In [21]:
from sklearn.metrics import accuracy_score, classification_report

In [23]:
accuracy = accuracy_score(
    all_labels,
    all_predictions,
)

print(f"Accuracy: {accuracy:.4f}")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Anxiety",
            "Depression",
            "Normal",
            "Suicidal",
        ],
        digits=4,
    )
)

Accuracy: 0.8431
              precision    recall  f1-score   support

     Anxiety     0.8072    0.9011    0.8516       799
  Depression     0.7638    0.7892    0.7763      2139
      Normal     0.9577    0.9490    0.9533      2723
    Suicidal     0.7768    0.7125    0.7432      1680

    accuracy                         0.8431      7341
   macro avg     0.8264    0.8379    0.8311      7341
weighted avg     0.8434    0.8431    0.8426      7341



In [24]:
print(len(train_df))
print(len(validation_df))
print(len(test_df))

34254
7340
7341


In [25]:
print(len(train_loader.dataset))
print(len(test_loader.dataset))

34254
7341


## Experiment 1 Result

Additional fine-tuning improved the model substantially.

Accuracy increased from 68.49% to 84.31%. The recall of the Suicidal class improved from 64.29% to 71.25%, while its F1-score increased from 58.60% to 74.32%.

The experiment suggests that the initial checkpoint had not yet converged and that additional fine-tuning allowed the model to learn more useful patterns. However, validation performance should also be monitored to detect potential overfitting.

# Experiment 1 Summary

## Objective

Evaluate whether additional fine-tuning improves the performance of the BERT classifier.

## Results

| Metric | Baseline | Experiment 1 |
|---------|----------|--------------|
| Accuracy | 68.49% | 84.31% |
| Suicidal Recall | 64.29% | 71.25% |
| Suicidal F1 | 58.60% | 74.32% |

## Interpretation

Additional fine-tuning significantly improved the model performance.

The largest improvements were observed across multiple classes while maintaining strong performance for the Normal class.

Most importantly, recall and F1-score for the Suicidal class improved, which is particularly valuable in mental health applications.

Future experiments will investigate learning rate, batch size, and class weighting.

## Limitations

The current experiment evaluates the impact of additional fine-tuning.

Although the results improved substantially, further experiments are required to evaluate:

- different learning rates,
- batch sizes,
- class weighting,
- early stopping,
- and validation performance across multiple training runs.

These experiments will be presented in the following notebook.